# Tier-1 v1.1 matrix (Colab)\n\nResumable: each cell writes `final_eval.json`; re-run skips completed cells.\nArtifacts persist on Google Drive under `beer_distribution_rl_artifacts/tier1_v11`.\n\nConstraints: local rewards A/B, no param sharing, unverified signals, honesty measured never rewarded.

In [ ]:
#@title 1) Mount Drive + clone/install\nfrom google.colab import drive\ndrive.mount("/content/drive")\n\nREPO_URL = "https://github.com/YOUR_ORG/beer_distribution_rl.git"  #@param {type:"string"}\nBRANCH = "main"  #@param {type:"string"}\nDRIVE_ROOT = "/content/drive/MyDrive/beer_distribution_rl_artifacts"\nOUT_DIR = f"{DRIVE_ROOT}/tier1_v11"\n\nimport pathlib, subprocess, sys\npathlib.Path(OUT_DIR).mkdir(parents=True, exist_ok=True)\nrepo = pathlib.Path("/content/beer_distribution_rl")\nif not repo.exists():\n    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo)])\nelse:\n    subprocess.check_call(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH])\n    subprocess.check_call(["git", "-C", str(repo), "checkout", BRANCH])\n    subprocess.check_call(["git", "-C", str(repo), "pull", "--ff-only"])\nsubprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[marl]"], cwd=repo)\nprint("OUT_DIR=", OUT_DIR)

In [ ]:
#@title 2) Device + matrix knobs\nimport torch\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nWORKERS = 2 if DEVICE == "cuda" else 8  #@param {type:"integer"}\nN_ENVS = 128 if DEVICE == "cuda" else 64  #@param {type:"integer"}\nTOTAL_TIMESTEPS = 400000  #@param {type:"integer"}\nprint(f"device={DEVICE} workers={WORKERS} n_envs={N_ENVS}")

In [ ]:
#@title 3) Dry-run cell count (pruned)\nimport subprocess, sys\nsubprocess.check_call([\n    sys.executable, "scripts/run_tier1_matrix.py", "--dry-run", "--out-dir", OUT_DIR\n], cwd="/content/beer_distribution_rl")

In [ ]:
#@title 4) Run matrix (resumable)\n# Re-run after Colab disconnect — completed cells are skipped via final_eval.json.\nimport subprocess, sys\nsubprocess.check_call([\n    sys.executable, "scripts/run_tier1_matrix.py",\n    "--out-dir", OUT_DIR,\n    "--device", DEVICE,\n    "--workers", str(WORKERS),\n    "--n-envs", str(N_ENVS),\n    "--total-timesteps", str(TOTAL_TIMESTEPS),\n    "--skip-existing",\n], cwd="/content/beer_distribution_rl")

In [ ]:
#@title 5) Quick status\nimport json, pathlib\nfrom collections import Counter\nidx = pathlib.Path(OUT_DIR) / "index.json"\nrows = json.loads(idx.read_text()) if idx.exists() else []\nprint("rows", len(rows), Counter(r.get("status") for r in rows))\na = [r for r in rows if r.get("regime") == "A"]\nb = [r for r in rows if r.get("regime") == "B"]\nprint("Regime A", len(a), "Regime B", len(b), "— D1 gap closable when both present")